In [ ]:
import pandas as pd
df = pd.read_csv("/content/Mental_health.csv", engine="python")
df

,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety
...,...,...,...
197199,19671,Cyclical depression Can’t get help because rea...,Depression
197200,2332,Childhood trauma When I was about 9-10 years o...,Depression
197201,19922,little poem or something like that.. I wrote t...,Depression
197202,20599,i think i'm depressed how do I fix it? I'm not...,Depression


In [ ]:
df.shape

(197204, 3)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197204 entries, 0 to 197203
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  197204 non-null  int64 
 1   statement   197134 non-null  object
 2   status      197204 non-null  object
dtypes: int64(1), object(2)
memory usage: 4.5+ MB


In [ ]:
df.drop(columns=['Unnamed: 0'],inplace=True)

In [ ]:
print(df.shape)
print('-'*50)
print(df.info())

(197204, 2)
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197204 entries, 0 to 197203
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   statement  197134 non-null  object
 1   status     197204 non-null  object
dtypes: object(2)
memory usage: 3.0+ MB
None


In [ ]:
df.value_counts('status')

,count
status,
Anxiety,49301
Depression,49301
Normal,49301
Suicidal,49301


In [ ]:
df.isnull().sum()

,0
statement,70
status,0


In [ ]:
df.duplicated().sum()

np.int64(43062)

In [ ]:
df = df.dropna(subset=['statement'])

In [ ]:
df.isnull().sum()

,0
statement,0
status,0


In [ ]:
df = df.drop_duplicates()

In [ ]:
df

,statement,status
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety
...,...,...
197199,Cyclical depression Can’t get help because rea...,Depression
197200,Childhood trauma When I was about 9-10 years o...,Depression
197201,little poem or something like that.. I wrote t...,Depression
197202,i think i'm depressed how do I fix it? I'm not...,Depression


In [ ]:
df.value_counts('status')

,count
status,
Anxiety,45403
Suicidal,39711
Depression,37201
Normal,31824


In [ ]:
# Check random raw text
df['statement'].sample(20, random_state=42)

,statement
35313,@gfrdtagon also again i hate making a comeback...
127010,Did cutting out caffeine give you more energy?...
36019,I just want to get my ass done Iâm tired of ...
117952,Always doubt what people think of me Someone d...
118578,has anyone with anxiety learned how to work wi...
112204,Does anyone else hear music? As of recent I've...
152636,The Saddest story in existence So basically I...
192966,Done. I’m sorry. I just need to say this out l...
193779,"Its obvious to me, but i dont know how, where,..."
114549,I fell like shit today Three days ago I finall...


In [ ]:
# Find contractions explicitly
df[df['statement'].str.contains(r"'", regex=True)].head(20)

,statement,status
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety
7,Have you ever felt nervous but didn't know why?,Anxiety
8,"I haven't slept well for 2 days, it's like I'm...",Anxiety
9,"I'm really worried, I want to cry.",Anxiety
10,"always restless every night, even though I don...",Anxiety
11,"I'm confused, I'm not feeling good lately. Eve...",Anxiety
14,Sometimes it's your own thoughts that make you...,Anxiety
15,"Every time I wake up, I'm definitely nervous a...",Anxiety
16,"I can only hope, even though I'm worried if it...",Anxiety


In [ ]:
# Text Cleaning
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    contractions = {
        "can't": "cannot",
        "won't": "will not",
        "'ll": " will",
        "'ve": " have",
        "'re": " are",
        "'d": " would",
        "'m": " am",
        "'s": " is",
        "n't": " not"
    }

    for pattern, replacement in contractions.items():
        text = re.sub(pattern, replacement, text)

    text = re.sub(r"[^a-z\s]", " ", text) # remove punctuation / special characters

    text = re.sub(r"\s+", " ", text).strip() # remove extra spaces


    return text

In [ ]:
df = df.copy()
df.loc[:, 'statement'] = df['statement'].apply(clean_text)

# remove empty / blank rows
df = df[df['statement'].notna() & (df['statement'].str.strip() != "")]

In [ ]:
df[df['statement'].str.contains("'", regex=False)]

,statement,status


In [ ]:
df

,statement,status
0,oh my gosh,Anxiety
1,trouble sleeping confused mind restless heart ...,Anxiety
2,all wrong back off dear forward doubt stay in ...,Anxiety
3,i have shifted my focus to something else but ...,Anxiety
4,i am restless and restless it is been a month ...,Anxiety
...,...,...
197199,cyclical depression can t get help because rea...,Depression
197200,childhood trauma when i was about years old my...,Depression
197201,little poem or something like that i wrote thi...,Depression
197202,i think i am depressed how do i fix it i am no...,Depression


# **FastText**

In [ ]:
# FastText format
df['status'] = "__label__" + df['status'] # FastText requires labels in this format: __label__class

df['fasttext'] = df['status'] + " " + df['statement'] # Combine label + text into single column
# Example: __label__Anxiety i feel nervous

In [ ]:
# Train-test split
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
# Save files
train['fasttext'].to_csv("train.txt", index=False, header=False) # Save training data in required format (no index, no header)

test['fasttext'].to_csv("test.txt", index=False, header=False) # Save test data for evaluation

In [ ]:
#Install FastText
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.3-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.3-py3-none-any.whl (313 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653980 sha256=b6cce0ba30e3cd8af38b532225e47d19030f9269962ad036d140e1f0857592f1
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
# Train model
model = fasttext.train_supervised(
    input="train.txt", # training data file # format: __label__class text
    epoch=60, # number of passes over data # higher = better learning, but slower
    lr=0.3,   # learning rate # controls how fast model updates weights
    wordNgrams=4,  # considers word combinations (up to 4 words) # helps capture phrases like "feel very anxious"
    dim=200, # size of word vectors (embedding size) # higher = more information, but larger model
    loss='ova' # one-vs-all loss for multi-class classification # treats each class separately
    # I used one-vs-all instead of softmax because in mental health data the same words and feelings can appear in multiple classes, so the model needs to evaluate each class independently rather than assume only one correct label.

)

In [ ]:
# Evaluate the model
result = model.test("test.txt")
print(result)

(30827, 0.797352969799202, 0.797352969799202)


In [ ]:
labels, probs = model.predict(["i feel very anxious and restless"])
print(labels, probs)

[['__label__Anxiety']] [array([1.00001], dtype=float32)]


In [ ]:
texts = [
    "i feel empty and tired",
    "i want to end my life",
    "i am feeling normal today"
]

labels, probs = model.predict(texts)
print(labels,probs)

[['__label__Depression'], ['__label__Suicidal'], ['__label__Normal']] [array([0.14415886], dtype=float32), array([1.00001], dtype=float32), array([0.0042088], dtype=float32)]


In [ ]:
tests = [
    "i feel empty and tired",
    "i feel hopeless",
    "i want to die",
    "i am just tired of everything"
]

labels, probs = model.predict(tests)
print(labels)

[['__label__Depression'], ['__label__Normal'], ['__label__Suicidal'], ['__label__Suicidal']]


In [ ]:
texts = test['statement'].tolist()
true_labels = test['status'].tolist()

pred_labels, _ = model.predict(texts)

In [ ]:
pred_labels = [label[0].replace("__label__", "") for label in pred_labels]
true_labels = [label.replace("__label__", "") for label in true_labels]

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(true_labels, pred_labels)
print(cm)

[[8173  606   75  269]
 [ 656 5462  100 1150]
 [ 136  193 5306  765]
 [ 302 1271  724 5639]]


In [ ]:
import pandas as pd

labels = ["Anxiety", "Depression", "Suicidal", "Normal"]
cm_df = pd.DataFrame(cm, index=labels, columns=labels)

print(cm_df)

            Anxiety  Depression  Suicidal  Normal
Anxiety        8173         606        75     269
Depression      656        5462       100    1150
Suicidal        136         193      5306     765
Normal          302        1271       724    5639


In [ ]:
print(classification_report(true_labels, pred_labels))

              precision    recall  f1-score   support

     Anxiety       0.88      0.90      0.89      9123
  Depression       0.73      0.74      0.73      7368
      Normal       0.86      0.83      0.84      6400
    Suicidal       0.72      0.71      0.72      7936

    accuracy                           0.80     30827
   macro avg       0.80      0.79      0.79     30827
weighted avg       0.80      0.80      0.80     30827



In [ ]:
model.save_model("mental_health_model.bin")

In [ ]:
# reload model if needed
model = fasttext.load_model("mental_health_model.bin")

In [ ]:
model.predict(["i feel anxious"])

([['__label__Anxiety']], [array([1.00001], dtype=float32)])

In [ ]:
from google.colab import files
files.download("mental_health_model.bin")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
print(os.path.getsize("mental_health_model.bin"))

1681220766


In [ ]:
import shutil
shutil.make_archive("model", 'zip', ".", "mental_health_model.bin")

'/content/model.zip'

In [ ]:
from google.colab import files
files.download("model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.quantize(input="train.txt", retrain=True)
model.save_model("model.ftz")

In [ ]:
import os
print(os.listdir())

['.config', 'model.ftz', '.ipynb_checkpoints', 'Mental_health.csv', 'model.zip', 'test.txt', 'train.txt', 'sample_data']


In [ ]:
from google.colab import files
files.download("model.ftz")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>